# LLM monitoring with Evidently

Reports on how answers behave, and drift against the week before. Needs the monitoring extra: `pip install -e ".[monitoring]"`.

See the [README](README.md) for the why and the how.

In [ ]:
# Run from anywhere inside the repository: find its root so `marginalia` and the test helpers import,
# and read .env (the same settings the services use).
import os, sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(ROOT)

In [ ]:
import random
from datetime import datetime, timedelta, timezone
from marginalia.monitoring import data, report
now = datetime.now(timezone.utc)
def row(i, tokens, status='done', supported=True):
    return {'created_at': now - timedelta(minutes=i), 'question': random.choice(['Who is Sonia?', 'Why did he confess?']),
            'answer': 'Sonia is Marmeladov\'s daughter [Part I, Chapter II].' if status == 'done' else None,
            'status': status, 'answered_by': 'agent' if status == 'done' else 'guard', 'supported': supported,
            'refused_reason': None if status == 'done' else 'off_topic', 'book': 'Crime and Punishment',
            'seconds': random.gauss(60, 10), 'total_tokens': tokens, 'prompt_tokens': tokens and tokens - 1500,
            'completion_tokens': tokens and 1500, 'tokens_per_second': random.gauss(60, 5), 'searches': random.randint(1, 4),
            'revisions': 0, 'citations': 1}
current = data.to_frame([row(i, int(random.gauss(9000, 900))) for i in range(40)] + [row(99, None, 'refused', None)])
reference = data.to_frame([row(i, int(random.gauss(6000, 600))) for i in range(150)])
current.head()

## The report
In a notebook, a snapshot renders as the interactive Evidently report.

In [ ]:
snapshot = report.build(current, reference)
snapshot

## Publishing
To a local workspace folder (viewable with `evidently ui --workspace ./evidently-workspace`), or to the Evidently UI service of the stack.

In [ ]:
from evidently.ui.workspace import Workspace
ws = Workspace.create('evidently-workspace')
proj = report.project(ws)
ws.add_run(proj.id, snapshot)
print(len(ws.list_runs(proj.id)), 'run(s) in the local workspace')

In [ ]:
# Needs the observability profile: one real report from the reporting views, into the Evidently UI.
from marginalia.config import Settings
from marginalia.monitoring.__main__ import run_once
run_once(Settings())